# Acme Procurement Desk — AI Procurement Decision Agent

**Competition task:** classify each supplier email thread as `ACCEPT`, `COUNTER`, `REJECT`, or `ESCALATE`, following the Acme procurement playbook. The output must be a two-column CSV: `thread_id,label`.

## What this notebook does
1. Loads the supplied agent bundle and procurement playbook.
2. Inspects the 2,000 supplier threads and the 40 worked examples.
3. Runs the deterministic procurement policy engine.
4. Retrieves similar historical examples for auditability.
5. Checks exact-match performance on the supplied worked examples.
6. Validates and saves `submission.csv` and `decision_audit.csv` in `/kaggle/working`.

> **Implementation note:** The submitted decision pipeline uses Python parsing and deterministic policy rules. Historical retrieval is text-similarity based. `llm_adapter.py` is an optional extension and is not called in this reproducible run; no API key or internet access is required. This is a policy-driven decision agent, not a separately trained LLM or a multi-agent LLM system.

## 1. Attach the project files to this Kaggle Notebook

Before running the notebook, add the uploaded Kaggle Dataset containing **`procurement_agent_bundle.zip`** using the notebook's **Add Input / Add Data** panel. The ZIP should be visible under `/kaggle/input/`.

The code below searches `/kaggle/input` for the ZIP and extracts it into the writable `/kaggle/working` directory.

In [1]:
from pathlib import Path
import zipfile
import shutil
import sys
import subprocess
import os

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
PROJECT_DIR = WORK_ROOT / "procurement_agent"

# Prefer the exact bundle name; otherwise use a single ZIP found in Kaggle inputs.
zip_candidates = list(INPUT_ROOT.rglob("procurement_agent_bundle.zip"))
if not zip_candidates:
    zip_candidates = list(INPUT_ROOT.rglob("*.zip"))

if zip_candidates:
    bundle_path = zip_candidates[0]
    print("Using bundle:", bundle_path)
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle_path, "r") as zf:
        zf.extractall(PROJECT_DIR)
else:
    # Also support adding an already-extracted project folder as a Kaggle Dataset.
    agent_files = list(INPUT_ROOT.rglob("run_agent.py"))
    if not agent_files:
        raise FileNotFoundError(
            "Could not find procurement_agent_bundle.zip or run_agent.py under /kaggle/input. "
            "Use Add Input to attach the project ZIP dataset."
        )
    source_dir = agent_files[0].parent
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    shutil.copytree(source_dir, PROJECT_DIR)

required = ["agent.py", "run_agent.py", "retrieval.py", "threads.csv", "sample_threads.csv", "sample_answers.csv"]
missing = [name for name in required if not (PROJECT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Project bundle is incomplete. Missing: {missing}")

print("Project directory:", PROJECT_DIR)
print("Top-level files:", sorted(p.name for p in PROJECT_DIR.iterdir() if p.is_file()))

Project directory: /kaggle/working/procurement_agent
Top-level files: ['README.md', 'agent.py', 'decision_audit.csv', 'llm_adapter.py', 'requirements.txt', 'retrieval.py', 'run_agent.py', 'sample_answers.csv', 'sample_submission.csv', 'sample_threads.csv', 'submission.csv', 'threads.csv', 'validation_report.txt']


## 2. Environment and dependencies

The Kaggle Python image normally includes pandas. The project only requires pandas for this run. The optional OpenAI package is not needed because the LLM adapter is disabled.

In [2]:
import pandas as pd

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
assert tuple(map(int, pd.__version__.split(".")[:2])) >= (2, 0), "Please use pandas 2.0 or newer."

Python: 3.13.18
pandas: 2.3.3


## 3. Inspect the input data

The main input is `threads.csv`. Each row represents one supplier negotiation thread. The playbook supplies the policy order, category prices, dated amendments, supplier status/contract overrides, and historical precedents.

In [3]:
threads_path = PROJECT_DIR / "threads.csv"
sample_threads_path = PROJECT_DIR / "sample_threads.csv"
sample_answers_path = PROJECT_DIR / "sample_answers.csv"

threads = pd.read_csv(threads_path).fillna("")
sample_threads = pd.read_csv(sample_threads_path).fillna("")
sample_answers = pd.read_csv(sample_answers_path).fillna("")

print("Competition threads:", threads.shape)
print("Worked-example threads:", sample_threads.shape)
print("Worked-example answers:", sample_answers.shape)
display(threads.head(5))

Competition threads: (2000, 8)
Worked-example threads: (40, 8)
Worked-example answers: (40, 2)


,thread_id,date,supplier,category,quantity,round,our_last_counter,supplier_message
0,T10001,2026-09-22,Ivory Events Catering,printer-paper,150,3,2400.0,"[Earlier message — Ivory Events Catering, 2026..."
1,T10002,2026-12-16,Monsoon Foods,headsets,100,2,3500.0,"Namaste team,\n\nFor the 100 headsets you requ..."
2,T10003,2026-12-15,Banyan Guarding,headsets,50,1,,"Hi team,\n\nFor the 50 headsets you requested,..."
3,T10004,2026-09-09,Garnet Print & Paper,courier,10,4,178.0,"[Earlier message — Garnet Print & Paper, 2026-..."
4,T10005,2026-09-29,Arvind Office Supplies,monitors,5,3,15250.0,"Good morning,\n\nFor the 5 monitors you reques..."


## 4. Decision architecture

The runner applies the playbook's ordered decision procedure. In brief, it checks missing/unsupported price information, supplier blacklist status, prompt injection, banned terms, normalized offer price and concession schedule, approval limit, and probation status. It applies amendments effective on or before the thread date and supplier contract overrides as encoded in the agent.

The retrieval component finds similar precedent/worked-example text and writes the matches and similarity scores to an audit CSV. Retrieved cases are supporting context; the official playbook remains authoritative.

In [4]:
# Load the category policies, amendments, and supplier profiles.
sys.path.insert(0, str(PROJECT_DIR))
import agent
from retrieval import retrieve_similar_cases

PLAYBOOK_DIR = PROJECT_DIR / "playbook" / "playbook"
categories, amendments, suppliers = agent.read_playbook(str(PLAYBOOK_DIR))
agent.enrich_category(categories, str(PLAYBOOK_DIR))

print("Categories loaded:", len(categories))
print("Dated amendments loaded:", len(amendments))
print("Supplier profiles loaded:", len(suppliers))
print("Example categories:", sorted(categories)[:8])

Categories loaded: 15
Dated amendments loaded: 9
Supplier profiles loaded: 40
Example categories: ['catering', 'cleaning', 'cloud-servers', 'courier', 'crm-licenses', 'design-software', 'headsets', 'laptops']


## 5. Validate against the 40 worked examples

The worked examples are a sanity check for the supplied policy implementation. They are not proof of hidden-test performance. We compare the complete label string, so the action, reason, and exact counter-price must all match.

In [5]:
sample_predictions = [
    agent.decision(row, categories, amendments, suppliers, str(PLAYBOOK_DIR))
    for _, row in sample_threads.iterrows()
]
comparison = sample_answers[["thread_id", "label"]].merge(
    pd.DataFrame({"thread_id": sample_threads["thread_id"], "predicted": sample_predictions}),
    on="thread_id", how="inner"
)
comparison["exact_match"] = comparison["label"] == comparison["predicted"]
correct = int(comparison["exact_match"].sum())
print(f"Worked-example exact match: {correct}/{len(comparison)} ({correct/len(comparison):.1%})")
if correct != len(comparison):
    display(comparison.loc[~comparison["exact_match"]])
else:
    print("All supplied worked examples match exactly.")

Worked-example exact match: 40/40 (100.0%)
All supplied worked examples match exactly.


## 6. Run the agent on all 2,000 competition threads

This executes the same batch runner included in the ZIP. The final CSV and audit are written directly to `/kaggle/working`, which is Kaggle's persistent notebook output directory.

In [6]:
submission_path = WORK_ROOT / "submission.csv"
audit_path = WORK_ROOT / "decision_audit.csv"

cmd = [
    sys.executable, "run_agent.py",
    "--threads", str(PROJECT_DIR / "threads.csv"),
    "--playbook", str(PLAYBOOK_DIR),
    "--output", str(submission_path),
    "--sample-threads", str(sample_threads_path),
    "--sample-answers", str(sample_answers_path),
    "--audit", str(audit_path),
]

completed = subprocess.run(cmd, cwd=str(PROJECT_DIR), text=True, capture_output=True)
print(completed.stdout)
if completed.stderr:
    print("STDERR:", completed.stderr)
if completed.returncode != 0:
    raise RuntimeError(f"Agent run failed with exit code {completed.returncode}")

assert submission_path.exists(), "submission.csv was not created"
assert audit_path.exists(), "decision_audit.csv was not created"

Created /kaggle/working/submission.csv with 2000 rows; labels structurally valid.
Worked-example exact match: 40/40 (100.0%)



## 7. Validate the submission format

Required format:

- Exactly two columns: `thread_id,label`.
- One output row for each of the 2,000 threads.
- Unique, non-empty thread IDs.
- Label format `ACTION:REASON:COUNTER_PRICE`.
- Counter price is an integer only for `COUNTER`; the final field is empty for other actions.

In [7]:
submission = pd.read_csv(submission_path, dtype=str).fillna("")
source_ids = threads["thread_id"].astype(str).tolist()

assert submission.columns.tolist() == ["thread_id", "label"]
assert len(submission) == len(threads) == 2000
assert submission["thread_id"].is_unique
assert submission["thread_id"].notna().all()
assert submission["thread_id"].astype(str).tolist() == source_ids

valid_reasons = {
    "ACCEPT": {"ACCEPTABLE_PRICE"},
    "COUNTER": {"COUNTER_OFFER"},
    "REJECT": {"ABOVE_WALKAWAY", "BLACKLISTED_SUPPLIER"},
    "ESCALATE": {"MISSING_INFO", "PROMPT_INJECTION", "BANNED_TERM", "OVER_APPROVAL_LIMIT"},
}
for label in submission["label"]:
    parts = label.split(":")
    assert len(parts) == 3, f"Bad label shape: {label}"
    action, reason, price = parts
    assert action in valid_reasons and reason in valid_reasons[action], f"Bad action/reason: {label}"
    if action == "COUNTER":
        assert price.isdigit(), f"Counter price must be a whole INR number: {label}"
    else:
        assert price == "", f"Non-counter label must end with an empty price field: {label}"

print("PASS — rows:", len(submission))
print("PASS — unique thread IDs:", submission.thread_id.nunique())
print("PASS — valid label structure and counter-price formatting")
print("Action counts:")
display(submission["label"].str.split(":").str[0].value_counts().rename_axis("action").to_frame("threads"))

PASS — rows: 2000
PASS — unique thread IDs: 2000
PASS — valid label structure and counter-price formatting
Action counts:


,threads
action,
ESCALATE,709
COUNTER,619
ACCEPT,345
REJECT,327


## 8. Review a sample of decisions and the retrieval audit

The audit file is for explainability/debugging. It records the chosen label and the most similar historical cases found by the local retrieval module. It is not part of the required competition submission.

In [8]:
audit = pd.read_csv(audit_path).fillna("")
print("Audit rows:", len(audit))
display(audit.head(10))
print("Submission preview:")
display(submission.head(10))

Audit rows: 2000


,thread_id,label,similar_cases,similarity_scores
0,T10001,REJECT:ABOVE_WALKAWAY:,T00014;T00028;T00020,0.5071;0.3795;0.36
1,T10002,ESCALATE:OVER_APPROVAL_LIMIT:,T00004;T00007;T00005,0.5328;0.5274;0.5204
2,T10003,ESCALATE:PROMPT_INJECTION:,T00007;T00005;T00004,0.4808;0.475;0.4461
3,T10004,REJECT:ABOVE_WALKAWAY:,T00028;T00014;T00022,0.4828;0.3643;0.3501
4,T10005,REJECT:ABOVE_WALKAWAY:,T00003;T00026;T00025,0.6;0.557;0.4801
5,T10006,ACCEPT:ACCEPTABLE_PRICE:,T00004;T00020;P-02,0.4049;0.3533;0.3445
6,T10007,ACCEPT:ACCEPTABLE_PRICE:,T00006;T00002;T00021,0.416;0.3693;0.3236
7,T10008,COUNTER:COUNTER_OFFER:62889,T00033;T00018;T00019,0.5417;0.4881;0.4754
8,T10009,ACCEPT:ACCEPTABLE_PRICE:,T00040;T00025;T00017,0.3333;0.3234;0.3175
9,T10010,COUNTER:COUNTER_OFFER:270,T00004;T00023;T00009,0.5274;0.478;0.4518


Submission preview:


,thread_id,label
0,T10001,REJECT:ABOVE_WALKAWAY:
1,T10002,ESCALATE:OVER_APPROVAL_LIMIT:
2,T10003,ESCALATE:PROMPT_INJECTION:
3,T10004,REJECT:ABOVE_WALKAWAY:
4,T10005,REJECT:ABOVE_WALKAWAY:
5,T10006,ACCEPT:ACCEPTABLE_PRICE:
6,T10007,ACCEPT:ACCEPTABLE_PRICE:
7,T10008,COUNTER:COUNTER_OFFER:62889
8,T10009,ACCEPT:ACCEPTABLE_PRICE:
9,T10010,COUNTER:COUNTER_OFFER:270


## 9. Final files

The required competition upload is **`/kaggle/working/submission.csv`**. The optional `decision_audit.csv` is useful for review but should not be uploaded as the prediction file.

For a classic Kaggle competition, download `submission.csv` from the notebook Output panel and upload it using the competition's **Submit Predictions** button. If the competition is explicitly a Code Competition, follow its notebook-only submission instructions instead.

In [9]:
from IPython.display import FileLink, display

print("Required submission file:")
display(FileLink(str(submission_path)))
print("Optional audit file:")
display(FileLink(str(audit_path)))

Required submission file:


/kaggle/working/submission.csv

Optional audit file:


/kaggle/working/decision_audit.csv